# 04 · Caching and reproducibility

**Story so far:** the processing pipeline works, but feature engineering is iterative:
we'll tweak one step and re-run the whole thing, many times a day. Caching is the
highest-leverage performance feature in Flyte: a cache hit costs milliseconds, while even
a trivial task execution costs pod-scheduling seconds. This chapter makes Review Radar's
featurization cheap to iterate and deterministic to reproduce.

**Learning goals**

1. Use every `Cache` behavior deliberately: `auto`, `override`, `disable`
2. Shape cache keys: `ignored_inputs`, `salt`, custom policies
3. Prevent duplicate concurrent work with `serialize=True`
4. Understand deterministic builds and when to pin `version_override`
5. Diagnose queueing delays, image-pull overhead, and use spot capacity safely

In [1]:
import flyte
from flyte import Cache

flyte.init_from_config()

env = flyte.TaskEnvironment(
    name="features",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
)

## 1. Cache behaviors

A task's cache key = **inputs + task version**. The `behavior` decides how that version
is derived:

| Behavior | Version comes from | Use for |
|---|---|---|
| `"auto"` | Hash of the task's code; edits invalidate automatically | Development, most pipelines |
| `"override"` | A string you pin (`version_override`) | Production stability across refactors |
| `"disable"` | None (never cached) | Side-effectful tasks (DB writes, notifications) |

In [ ]:
@env.task(cache="auto")                    # shorthand for Cache(behavior="auto")
async def featurize(text: str) -> dict:
    print("cache MISS, computing features")
    words = text.lower().split()
    return {"n_words": len(words), "n_unique": len(set(words))}


run = await flyte.run.aio(featurize, text="solid espresso machine works as advertised")
await run.wait.aio()

# Run again with the same input: cache hit, completes ~instantly.
run2 = await flyte.run.aio(featurize, text="solid espresso machine works as advertised")
await run2.wait.aio()
print(run2.url)

In [ ]:
# Production: survive refactors, invalidate only when you bump the version
@env.task(cache=Cache(behavior="override", version_override="v3"))
async def stable_featurize(text: str) -> dict:
    return {"n_chars": len(text)}


# Side effects: never cache
@env.task(cache=Cache(behavior="disable"))
async def notify_data_team(message: str) -> None:
    print(f"sending: {message}")

## 2. Shaping the cache key

The cache key is the task version plus its declared **inputs**. `ignored_inputs` drops
named inputs from that key, so two calls that differ only in those inputs still hit the
cache. Use it for inputs that do not change the output, such as a random seed or a
timestamp you pass in.

In [ ]:
# salt: namespace the cache, e.g. per experiment; bump the salt to force a clean slate
@env.task(cache=Cache(behavior="auto", salt="feature_set_q3"))
async def experimental_features(text: str) -> dict:
    return {"vowels": sum(text.count(v) for v in "aeiou")}


# serialize=True: if two runs request the same (task, inputs) simultaneously,
# one executes and the other waits for its result. Essential for the expensive
# vocabulary-build step that several experiments share.
@env.task(cache=Cache(behavior="auto", serialize=True))
async def build_vocabulary(corpus_version: str) -> dict:
    return {"vocab": f"built-for-{corpus_version}"}

**Custom cache policies** let the version depend on something external (e.g. a dataset
registry) so cache invalidation follows your data, not your code:

```python
from flyte._cache import CachePolicy

class CorpusVersionPolicy(CachePolicy):
    def get_version(self, salt: str, params) -> str:
        return f"{salt}_{lookup_corpus_version()}"

@env.task(cache=Cache(behavior="auto", policies=[CorpusVersionPolicy()]))
async def corpus_dependent(path: str) -> dict: ...
```

Force a re-execution without touching any definitions (e.g. upstream data corrected):

In [ ]:
run = flyte.with_runcontext(overwrite_cache=True).run(
    featurize, text="solid espresso machine works as advertised"
)
print(run.url)

## 3. Deterministic builds and reproducibility

Three layers decide whether "the same pipeline" really is the same:

1. **Image**. `flyte.Image` builds are content-addressed: same definition → same image.
   Pin every package version in `with_pip_packages(...)` (as these notebooks do); an
   unpinned `"pandas"` resolves differently over time and silently changes your runtime.
2. **Task version**. `cache="auto"` re-versions on any code change, including comments.
   For production pipelines pin `Cache(behavior="override", version_override="vN")` and
   treat bumping `vN` as a deliberate, reviewable act.
3. **Inputs**. Files/dataframes are content-addressed in the object store; two runs on
   the same inputs are byte-identical from Flyte's perspective.

**Recommended lifecycle:** `cache="auto"` while developing → switch hot production tasks
to `override` with an explicit version at release time. That's also the cleanest answer
to "why did yesterday's run cache-miss?": with `override`, it can't, until you bump it.

> 💬 **Discuss:** which of the customer's production tasks should move from `auto` to
> `override`, and what would their process be for bumping the version: PR review?
> release checklist?

## 4. Performance clinic

Where the time actually goes in a task execution, and the lever for each:

| Stage | Symptom | Lever |
|---|---|---|
| **Queueing** | Run pending, no capacity | `Timeout(max_queued_time=...)` to fail fast; capacity (platform side); `queue=` targeting |
| **Image pull** | Slow container start on new nodes | Smaller images; reusable containers ([05](./05-reusable-containers.ipynb)) |
| **Cold start** | Every tiny task costs ~10-60s | Reusable containers ([05](./05-reusable-containers.ipynb)) |
| **Execution** | Task slow | Right-size resources; fan out ([03](./03-processing-at-scale.ipynb)) |
| **I/O** | Slow start/end of task | Keep large data in `File`/`Dir`/`DataFrame` ([02](./02-data-flow.ipynb)), not inline |

Two levers you control from code:

In [ ]:
# queue: target a scheduling queue the deployment exposes (see appendix A)
# interruptible: allow spot/preemptible capacity; big savings for retry-tolerant work
spot_env = flyte.TaskEnvironment(
    name="spot_features",
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    interruptible=True,          # spot eviction does not consume your retry budget
    # queue="batch-queue",       # uncomment with a queue name for this deployment
)


@spot_env.task(retries=3)
async def featurize_shard(shard_id: int) -> str:
    return f"features for shard {shard_id}"


# Critical steps opt back out at the task level:
@spot_env.task(interruptible=False)
async def persist_feature_set(uri: str) -> str:
    return f"persisted {uri}"

### Under the hood

`interruptible=True` schedules onto the deployment's spot/preemptible capacity (pool
wiring is platform-side; see the *GPU / spot capacity* row in appendix A). Cloud providers give spot
VMs a short termination notice; Flyte reschedules the attempt on eviction without burning
user retries. Pair spot workers with `@flyte.trace` checkpoints
([03](./03-processing-at-scale.ipynb) §5) so long featurization jobs resume mid-flight
instead of restarting.

**Story checkpoint:** feature iteration is now cheap (cache hits) and releases are
reproducible (pinned versions). Time to put the corpus through a model, every review,
efficiently.

## Further reading

- Union docs: [Caching](https://www.union.ai/docs/v2/union/user-guide/task-configuration/caching/)
- Next: [05-reusable-containers](./05-reusable-containers.ipynb): kills the cold-start
  row of the table above